# Bayesian optimization with BoTorch: a first walkthrough

This notebook builds one **single-objective Bayesian optimization (BO)** iteration. We start with the core workflow; constraints, noisy objectives, batches, and multi-objective BO can come later. This will be connected to self-driving laboratory (SDL).

## The SDL loop

1. Collect evaluated inputs `train_X` and outcomes `train_Y`.
2. Fit a Gaussian process (GP) surrogate.
3. Use its posterior to estimate outcomes and uncertainty.
4. Score possible experiments with an acquisition function.
5. Optimize that function inside the allowed search space.
6. Run the proposed experiment, add its measured result, and repeat.

```text
observations -> GP surrogate -> acquisition -> next experiment
      ^                                         |
      +------------ add measured outcome --------+
```

For `n` experiments, `d` input variables, and one outcome, `train_X` has shape `n x d` and `train_Y` has shape `n x 1`.

## Example problem

We will maximize a simple one-variable objective `y = f(x)`, with `x` in `[0, 1]`. This small synthetic example keeps the first model easy to inspect; the same workflow extends to multiple input variables by using one column per variable.

For a real experiment, each input column might represent a formulation variable (for example, concentration or solvent fraction), and the outcome might be conductivity, yield, or another measured quantity.

## 1. Prepare the observations

Each row of `train_X` is one evaluated input, and the matching row of `train_Y` is its observed outcome. The inputs below are already normalized to the unit interval, which is a good default for GP models. For a physical variable with bounds `[a, b]`, map it to `(x - a) / (b - a)` and keep those bounds to convert the recommendation back later.

We use double precision for this small GP example and generate deterministic toy observations.

In [63]:
import torch

# GP calculations are generally more stable in double precision.
torch.set_default_dtype(torch.double)

train_X = torch.linspace(0.0, 1.0, 7).unsqueeze(-1)
train_Y = torch.sin(2 * torch.pi * train_X)

print("train_X shape:", train_X.shape)  # n x d = 7 x 1
print("train_Y shape:", train_Y.shape)  # n x 1

train_X shape: torch.Size([7, 1])
train_Y shape: torch.Size([7, 1])


## 2. Build the GP surrogate

A Gaussian process places a distribution over plausible objective functions. After seeing the data, its posterior gives a predictive mean and variance at any input.

`SingleTaskGP` is a standard choice for one outcome. In this BoTorch version, it uses a constant mean, a scaled Matern kernel, a Gaussian likelihood, and standardizes the outcome by default. The input in this example is already normalized; BoTorch does not automatically rescale the physical input space for us.

In [64]:
from botorch.models import SingleTaskGP

model = SingleTaskGP(train_X, train_Y)
print(model)

SingleTaskGP(
  (likelihood): GaussianLikelihood(
    (noise_covar): HomoskedasticNoise(
      (noise_prior): GammaPrior()
      (raw_noise_constraint): GreaterThan(1.000E-04)
    )
  )
  (mean_module): ConstantMean()
  (covar_module): ScaleKernel(
    (base_kernel): MaternKernel(
      (lengthscale_prior): GammaPrior()
      (raw_lengthscale_constraint): Positive()
    )
    (outputscale_prior): GammaPrior()
    (raw_outputscale_constraint): Positive()
  )
)


/Users/benediktusmadika/Desktop/AI learning/botorch_sdl/lib/python3.9/site-packages/botorch/models/utils/assorted.py:202: InputDataWarning: Input data is not standardized (mean = tensor([-3.0086e-17]), std = tensor([0.7071])). Please consider scaling the input to zero mean and unit variance.
  warnings.warn(msg, InputDataWarning)


### A few GP parts to recognize

- **Mean function:** the baseline trend before considering nearby observations.
- **Kernel:** describes how similarity between inputs relates to similarity between outcomes. The default scaled Matern kernel is a flexible smoothness assumption.
- **Lengthscale and outputscale:** learned kernel parameters describing how quickly the function varies and its overall variation.
- **Gaussian likelihood:** accounts for observation noise; it has a learned noise level.

The model is constructed, but its hyperparameters still need to be fit to the observations.

## 3. Fit the model

The marginal log likelihood measures how well the GP hyperparameters explain the observed data. Fitting maximizes this quantity (equivalently, minimizes its negative); BoTorch chooses and runs the appropriate optimizer for this model.

In [65]:
from gpytorch.mlls import ExactMarginalLogLikelihood
from botorch.fit import fit_gpytorch_mll

mll = ExactMarginalLogLikelihood(model.likelihood, model)
fit_gpytorch_mll(mll);

## 4. Inspect the posterior

For candidate inputs `X`, the GP posterior summarizes predicted outcomes given the data. Its mean is the current best estimate; its variance describes predictive uncertainty.

BoTorch model inputs have shape `batch_shape x q x d`: `q` is the number of points considered jointly and `d` is the number of input variables. Here, three points have shape `3 x 1`.

In [66]:
test_X = torch.tensor([[0.25], [0.50], [0.75]])

In [72]:
posterior = model.posterior(test_X)
print("Posterior mean:\\n", posterior.mean)
print("Posterior variance:\\n", posterior.variance)

Posterior mean:\n tensor([[ 1.0047e+00],
        [-6.8797e-16],
        [-1.0047e+00]], grad_fn=<UnsqueezeBackward0>)
Posterior variance:\n tensor([[0.0055],
        [0.0024],
        [0.0055]], grad_fn=<UnsqueezeBackward0>)


## 5. Score candidates with an acquisition function

An acquisition function uses the model's predictions to score where it may be valuable to run the next experiment. It balances predicted performance with uncertainty, according to the chosen rule; it does not run the experiment.

For a maximization problem, **Expected Improvement (EI)** favors candidates expected to exceed the best observed outcome. **LogEI** is a numerically more stable version, often preferred when optimizing EI. Acquisition functions expect candidates shaped `batch_shape x q x d`. To score each row of `test_X` as a separate one-point batch, insert the `q` dimension with `unsqueeze(-2)`.

In [68]:
from botorch.acquisition import PosteriorMean

# Posterior mean alone is a simple baseline; it does not reward uncertainty.
posterior_mean_acq = PosteriorMean(model=model, maximize=True)
mean_scores = posterior_mean_acq(test_X.unsqueeze(-2))
print("Posterior-mean scores:", mean_scores)

Posterior-mean scores: tensor([ 1.0047e+00, -6.8797e-16, -1.0047e+00], grad_fn=<SqueezeBackward1>)


In [69]:
from botorch.acquisition import ExpectedImprovement

best_f = train_Y.max()
ei = ExpectedImprovement(model=model, best_f=best_f)
ei_scores = ei(test_X.unsqueeze(-2))
print("Best observed outcome:", best_f)
print("EI scores:", ei_scores)

Best observed outcome: tensor(0.8660)
EI scores: tensor([ 1.3952e-01,  8.4129e-72, 1.8458e-143], grad_fn=<MulBackward0>)


For maximization, EI is

```math
EI(x) = E[max(f(x) - f_best, 0)]
```

where `f_best = max(train_Y)`. A high score means the model expects more improvement, accounting for both the predictive mean and uncertainty. Posterior mean alone can miss uncertain but promising regions; EI addresses that exploration/exploitation trade-off.

In [70]:
from botorch.acquisition import LogExpectedImprovement

acq = LogExpectedImprovement(model=model, best_f=best_f)
log_ei_scores = acq(test_X.unsqueeze(-2))
print("LogEI scores:", log_ei_scores)

LogEI scores: tensor([  -1.9695, -163.6564, -328.6568], grad_fn=<AddBackward0>)


## 6. Optimize the acquisition function

Rather than score only a hand-picked grid, `optimize_acqf` searches the continuous bounded domain for a high-scoring candidate. Its `bounds` must use the same normalized coordinates as the model's training inputs.

Here `q=1` asks for one next experiment. Acquisition optimization is non-convex, so BoTorch uses sampled initial points and multiple local restarts.

In [71]:
from botorch.optim import optimize_acqf

bounds = torch.tensor([[0.0], [1.0]])  # lower and upper bounds, one row per input dimension

candidate, acq_value = optimize_acqf(
    acq_function=acq,
    bounds=bounds,
    q=1,
    num_restarts=10,
    raw_samples=100,
)

print("Proposed normalized input:", candidate)
print("LogEI at proposal:", acq_value)

Proposed normalized input: tensor([[0.2498]])
LogEI at proposal: tensor(-1.9695)


### What the optimizer is doing

1. Sample possible starting points within `bounds`.
2. Select promising starts.
3. Locally optimize the acquisition function from multiple starts.
4. Return the candidate with the best acquisition value found.

The returned candidate is a **proposal**, not a measured result. In a real workflow, map it back to physical units, run the experiment, record its outcome, append the new `(X, Y)` pair, then refit and optimize again.

## 7. The next BO iteration

After evaluating the proposed experiment at `candidate`, suppose the measured outcome is `new_Y`. The basic loop is:

```python
train_X = torch.cat([train_X, candidate])
train_Y = torch.cat([train_Y, new_Y])
```

Then refit the GP using the updated data and find the next proposal. In practice, also track units, experimental failures, replicate measurements, and whether the objective is to maximize or minimize.

### Key takeaways

- The GP models the unknown objective and its uncertainty.
- The acquisition function turns that posterior into a search policy.
- Acquisition optimization proposes where to evaluate next.
- Only a real measurement (or a simulator result) provides the next observed outcome.

# Full first BO calculation


In [73]:
import torch

from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_mll
from gpytorch.mlls import ExactMarginalLogLikelihood

from botorch.acquisition import LogExpectedImprovement
from botorch.optim import optimize_acqf


# ============================================================
# 1. Configuration
# ============================================================

torch.set_default_dtype(torch.double)


# ============================================================
# 2. Initial experimental data
# ============================================================

train_X = torch.tensor([
    [0.0000],
    [0.1667],
    [0.3333],
    [0.5000],
    [0.6667],
    [0.8333],
    [1.0000],
])

train_Y = torch.tensor([
    [0.0770],
    [0.8514],
    [0.7571],
    [0.0284],
    [-0.9203],
    [-0.9360],
    [0.0202],
])


# ============================================================
# 3. Construct GP
# ============================================================

model = SingleTaskGP(
    train_X,
    train_Y,
)


# ============================================================
# 4. Construct marginal log likelihood
# ============================================================

mll = ExactMarginalLogLikelihood(
    model.likelihood,
    model,
)


# ============================================================
# 5. Fit GP hyperparameters
# ============================================================

fit_gpytorch_mll(mll)


# ============================================================
# 6. Current best observation
# ============================================================

best_f = train_Y.max()

print("Current best:")
print(best_f)


# ============================================================
# 7. Construct acquisition function
# ============================================================

acq = LogExpectedImprovement(
    model=model,
    best_f=best_f,
)


# ============================================================
# 8. Define search space
# ============================================================

bounds = torch.tensor([
    [0.0],
    [1.0],
])


# ============================================================
# 9. Optimize acquisition function
# ============================================================

candidate, acq_value = optimize_acqf(
    acq_function=acq,
    bounds=bounds,
    q=1,
    num_restarts=10,
    raw_samples=100,
)


# ============================================================
# 10. Proposed next experiment
# ============================================================

print("Next candidate:")
print(candidate)

print("Acquisition value:")
print(acq_value)

/Users/benediktusmadika/Desktop/AI learning/botorch_sdl/lib/python3.9/site-packages/botorch/models/utils/assorted.py:202: InputDataWarning: Input data is not standardized (mean = tensor([-0.0175]), std = tensor([0.7102])). Please consider scaling the input to zero mean and unit variance.
  warnings.warn(msg, InputDataWarning)


Current best:
tensor(0.8514)
Next candidate:
tensor([[0.2346]])
Acquisition value:
tensor(-2.5271)


# BoTorch has only proposed the next experiment.
It has not measured the objective.
In a real self-driving laboratory (SDL) system: candidate would be sent to your automated laboratory.

```text
candidate
   │
   ▼
SDL formulation
   │
   ▼
mixing
   │
   ▼
electrolyte preparation
   │
   ▼
cell fabrication
   │
   ▼
electrochemical measurement
   │
   ▼
ionic conductivity
   │
   ▼
new_Y

In [ ]:
# Suppose SDL measures:
new_X = candidate
new_Y = torch.tensor([[1.72]])
train_X = torch.cat([
    train_X,
    new_X,
])

train_Y = torch.cat([
    train_Y,
    new_Y,
])

# Now your dataset contains one more experiment.

# you construct the model again:

model = SingleTaskGP(
    train_X,
    train_Y,
)

mll = ExactMarginalLogLikelihood(
    model.likelihood,
    model,
)

fit_gpytorch_mll(mll)

# Then construct the acquisition function again, and optimize it again, to propose the next experiment.
best_f = train_Y.max()

acq = LogExpectedImprovement(
    model=model,
    best_f=best_f,
)
 # candidate, acq_value = optimize_acqf(...)

# The actual sequential BO loop

```python
for iteration in range(N_ITERATIONS):

    # 1. Fit surrogate
    model = SingleTaskGP(
        train_X,
        train_Y,
    )

    mll = ExactMarginalLogLikelihood(
        model.likelihood,
        model,
    )

    fit_gpytorch_mll(mll)

    # 2. Construct acquisition
    best_f = train_Y.max()

    acq = LogExpectedImprovement(
        model=model,
        best_f=best_f,
    )

    # 3. Optimize acquisition
    candidate, acq_value = optimize_acqf(
        acq_function=acq,
        bounds=bounds,
        q=1,
        num_restarts=10,
        raw_samples=100,
    )

    # 4. REAL EXPERIMENT
    new_Y = run_experiment(candidate)

    # 5. Update dataset
    train_X = torch.cat([
        train_X,
        candidate,
    ])

    train_Y = torch.cat([
        train_Y,
        new_Y,
    ])


This is the fundamental architecture you are ultimately going to use for SDL.

In [ ]:
# very useful insepections

model.eval()

test_X = torch.tensor([
    [0.1],
    [0.25],
    [0.5],
    [0.75],
    [0.9],
])

posterior = model.posterior(test_X)

print("Mean:")
print(posterior.mean)

print("\nVariance:")
print(posterior.variance)

print("\nStandard deviation:")
print(posterior.variance.sqrt())

```text
                  EXPERIMENTAL DATA
                         │
                    train_X/train_Y
                         │
                         ▼
                  ┌──────────────┐
                  │ SingleTaskGP │
                  └──────┬───────┘
                         │
                         ▼
               ExactMarginalLogLikelihood
                         │
                         ▼
                 fit_gpytorch_mll
                         │
                         ▼
                   FITTED GP MODEL
                         │
                         ▼
                  model.posterior(X)
                         │
                         ▼
                ┌─────────────────┐
                │ μ(X), σ²(X), K │
                └────────┬────────┘
                         │
                         ▼
                 LogExpectedImprovement
                         │
                         ▼
                         α(X)
                         │
                         ▼
                   optimize_acqf
                         │
                         ▼
                    candidate
                         │
                         ▼
                  REAL EXPERIMENT
                         │
                         ▼
                     new_Y
                         │
                         ▼
                  append to dataset
                         │
                         └──────────────► repeat

# One important thing for your future SDL implementation

```python
while True:
    model = ...
    candidate = ...
    experiment(candidate)


```text
with everything mixed into one giant function.

Instead, eventually we should have explicit components:

```python
initialize_dataset()
fit_model()
build_acquisition()
optimize_acquisition()
run_sdl_experiment()
update_dataset()
evaluate_iteration()

This will become particularly important when we introduce:

- noisy measurements
- constraints
- failed experiments
- categorical electrolyte components
- mixture constraints
- multi-objective optimization
- multi-fidelity experiments
- neural molecular representations
- asynchronous SDL
- pending experiments
- experiment costs

The most important equation to remember is:

$$
\boxed{
x_{t+1}
=
\arg\max_{x\in\mathcal X}
\alpha\left(x\mid D_t\right)
}
$$

where:

$$
D_t=\{(x_i,y_i)\}_{i=1}^{t}.
$$

The GP transforms:

$$
D_t \rightarrow p(f(x)\mid D_t),
$$

and the acquisition function transforms:

$$
p(f(x)\mid D_t) \rightarrow \alpha(x).
$$

Then `optimize_acqf` solves the final maximization.